# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR⁲ dataset using the `mlcroissant` library. All exploration, processing, and EDA refer to record sets, fields, and columns using their Croissant `@id` fields to ensure clarity, reproducibility, and cross-tool compatibility.

### Dataset Source
The dataset source is specified by the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR⁲ dataset's schema and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
from pprint import pprint

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("{}:\n{}".format(metadata.name, metadata.description))

## 2. Data Overview
Review available record sets, their fields, and Croissant `@id`s. Identifying these is crucial for accessing, extracting, and transforming data.

*Record sets* are major data tables; *fields* correspond to columns. Each Croissant entity is referenced below by its `@id`.

In [ ]:
# List all record sets with their @id and name
print("\nRecord sets in the dataset:")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id} | name: {rs.name}")
    record_sets.append(rs.id)

# Show fields for each record set
for rs in dataset.metadata.record_sets:
    print(f"\nFields for record set: {rs.name} (@id: {rs.id})")
    for field in rs.fields:
        print(f"  - Field @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Extract tabular data from each record set by `@id`. We load the records into pandas DataFrames, which sets up for analysis.

Below, `record_sets` is a list of all record set `@id`s as gathered in the previous cell.

In [ ]:
dataframes = dict()

for record_set_id in record_sets:
    # Fetch records for the record set using its @id
    records = list(dataset.records(record_set=record_set_id))
    try:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded record set '@id': {record_set_id} with shape {dataframes[record_set_id].shape}")
        print(f"Columns (@id): {list(dataframes[record_set_id].columns)[:10]}" + ("..." if dataframes[record_set_id].shape[1]>10 else ""))
    except Exception as e:
        print(f"Could not load record set '@id': {record_set_id}: {e}")

# Print the first rows of the first available recordset
if len(dataframes) > 0:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nSample records from '@id': {first_rs_id}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply classic processing steps: filtering a numeric field, normalizing that field, and grouping by a key field. All field, record set, and column references use Croissant `@id`s.

*First,* choose a record set and a numeric field (`@id`) to operate on. Replace example IDs below as needed for your data if the suggested ones don't exist.

In [ ]:
# Example: Assume you have a record set and field to analyze
example_record_set = None
numeric_field_id = None
group_field_id = None

for rs in dataset.metadata.record_sets:
    for field in rs.fields:
        # Pick the first numeric field found
        if field.data_type in ("Float", "Integer", "Number") and numeric_field_id is None:
            example_record_set = rs.id
            numeric_field_id = field.id
        # Pick a categorical/text field for grouping if available
        if field.data_type in ("Text", "String") and group_field_id is None:
            group_field_id = field.id
    if example_record_set and numeric_field_id and group_field_id:
        break

if not example_record_set:
    raise ValueError("No suitable numeric field found in any record set.")

print(f"Analyzing record set '@id': {example_record_set}")
print(f"Numeric field '@id': {numeric_field_id}")
if group_field_id:
    print(f"Grouping field '@id': {group_field_id}")

df = dataframes[example_record_set]

# Check type for filtering
try:
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = numeric_series.mean() if numeric_series.notnull().sum() > 0 else 0
    filtered_df = df[numeric_series > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric column
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series - numeric_series.mean()) / numeric_series.std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id if available and field exists in DataFrame
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
except Exception as e:
    print(f"EDA step failed: {e}")

## 5. Visualization
Visualize the distribution of a numeric field and its relationship with a group/categorical field. All plots refer to fields by their Croissant `@id`s. Adjust the IDs as required for your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot by group field (if present)
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded the FAIR⁲ dataset via its Croissant schema URL with `mlcroissant`
- Explored available record sets, fields, and their Croissant `@id`s
- Extracted data tables and referenced all entities by their `@id`
- Performed EDA and basic statistical summarization using explicit `@id` field references
- Visualized distributions and group relationships using Croissant IDs for scientific reproducibility

Refer to the original metadata schema for further provenance, citations, and recommended data uses.